# Bengaluru Brain: M1 Traffic Module - Traffic Prediction Pipeline

## Overview
This notebook demonstrates the complete traffic prediction pipeline for Bangalore using Graph Neural Networks (GNN). The pipeline processes road network data, generates synthetic traffic features, trains a GNN model, and visualizes predictions on an interactive map.

## Pipeline Steps:
1. **Data Collection**: Download Bangalore road network using OSMnx
2. **Feature Engineering**: Generate synthetic traffic features
3. **Graph Processing**: Convert to PyTorch Geometric format
4. **Model Training**: Train GNN for congestion prediction
5. **Prediction & Visualization**: Generate predictions and create interactive maps

---

## 1. Environment Setup & Library Installation

In [ ]:
# Install required libraries
!pip install osmnx geopandas torch torch-geometric pandas numpy matplotlib networkx scikit-learn folium

# Import libraries
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import osmnx as ox
import networkx as nx
import folium
from folium.plugins import HeatMap
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Data Collection - Bangalore Road Network

In [ ]:
# Download Bangalore road network
print("Downloading Bangalore road network...")
G = ox.graph_from_place("Bangalore, India", network_type='drive')

# Display graph statistics
print(f"Graph downloaded successfully!")
print(f"Number of nodes: {G.number_of_nodes():,}")
print(f"Number of edges: {G.number_ofedges():,}")

# Save graph
ox.save_graphml(G, filepath="../data/bangalore.graphml")
print("Graph saved as bangalore.graphml")

In [ ]:
# Visualize the road network
fig, ax = ox.plot_graph(G, figsize=(12, 12), node_size=0, edge_linewidth=0.5, 
                       show=False, close=False)
plt.title("Bangalore Road Network", fontsize=16)
plt.savefig("../output/bangalore_road_network.png", dpi=300, bbox_inches='tight')
plt.show()

print("Road network visualization saved!")

## 3. Data Processing & Feature Engineering

In [ ]:
# Convert graph to CSV format
print("Converting graph to CSV format...")

# Extract nodes data
nodes_data = []
for node, data in G.nodes(data=True):
    node_info = {
        'node_id': node,
        'x': data.get('x', 0),
        'y': data.get('y', 0),
        'street_count': data.get('street_count', 0)
    }
    nodes_data.append(node_info)

nodes_df = pd.DataFrame(nodes_data)
nodes_df.to_csv('../data/nodes.csv', index=False)
print(f"Nodes CSV saved: {len(nodes_df):,} nodes")

# Extract edges data
edges_data = []
for u, v, data in G.edges(data=True):
    edge_info = {
        'source': u,
        'target': v,
        'length': data.get('length', 0),
        'maxspeed': data.get('maxspeed', None),
        'highway': data.get('highway', None),
        'name': data.get('name', None),
        'oneway': data.get('oneway', False)
    }
    edges_data.append(edge_info)

edges_df = pd.DataFrame(edges_data)
edges_df.to_csv('../data/edges.csv', index=False)
print(f"Edges CSV saved: {len(edges_df):,} edges")

In [ ]:
# Generate synthetic traffic features
print("Generating synthetic traffic features...")

np.random.seed(42)  # For reproducibility

# Generate traffic speed (20-60 km/h)
traffic_speed = []
for street_count in nodes_df['street_count']:
    base_speed = np.random.uniform(20, 60)
    speed_adjustment = -min(street_count * 2, 20)
    final_speed = max(20, min(60, base_speed + speed_adjustment))
    traffic_speed.append(final_speed)

# Generate rainfall (0-10 mm)
rainfall = np.random.uniform(0, 10, len(nodes_df))

# Generate road capacity (1-4 lanes)
road_capacity = []
for street_count in nodes_df['street_count']:
    if street_count <= 2:
        capacity = np.random.choice([1, 2], p=[0.7, 0.3])
    elif street_count <= 4:
        capacity = np.random.choice([2, 3], p=[0.6, 0.4])
    else:
        capacity = np.random.choice([3, 4], p=[0.4, 0.6])
    road_capacity.append(capacity)

# Create traffic nodes dataframe
traffic_nodes_df = nodes_df.copy()
traffic_nodes_df['traffic_speed'] = traffic_speed
traffic_nodes_df['rainfall'] = rainfall
traffic_nodes_df['road_capacity'] = road_capacity

# Generate congestion scores (target variable)
congestion_scores = []
for idx, row in traffic_nodes_df.iterrows():
    speed_factor = (60 - row['traffic_speed']) / 40
    rainfall_factor = row['rainfall'] / 10
    capacity_factor = (4 - row['road_capacity']) / 3
    street_factor = min(row['street_count'] / 10, 1)
    
    congestion = (speed_factor * 0.4 + rainfall_factor * 0.2 + 
                  capacity_factor * 0.2 + street_factor * 0.2)
    congestion += np.random.normal(0, 0.1)
    congestion = max(0, min(1, congestion))
    
    congestion_scores.append(congestion)

traffic_nodes_df['congestion_score'] = congestion_scores
traffic_nodes_df.to_csv('../data/traffic_nodes.csv', index=False)
print(f"Traffic features generated for {len(traffic_nodes_df):,} nodes")

# Display statistics
print("\nTraffic Features Statistics:")
print(traffic_nodes_df[['traffic_speed', 'rainfall', 'road_capacity', 'congestion_score']].describe())

## 4. Graph Neural Network Model Architecture

In [ ]:
class TrafficGNN(nn.Module):
    """Graph Neural Network for traffic congestion prediction"""
    
    def __init__(self, input_dim=3, hidden_dim1=16, hidden_dim2=8, output_dim=1, dropout=0.2):
        super(TrafficGNN, self).__init__()
        
        # Graph convolution layers
        self.conv1 = GCNConv(input_dim, hidden_dim1)
        self.conv2 = GCNConv(hidden_dim1, hidden_dim2)
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_dim2, hidden_dim2 // 2)
        self.fc2 = nn.Linear(hidden_dim2 // 2, output_dim)
        
        # Regularization
        self.dropout = nn.Dropout(dropout)
        self.bn1 = nn.BatchNorm1d(hidden_dim1)
        self.bn2 = nn.BatchNorm1d(hidden_dim2)
        self.bn3 = nn.BatchNorm1d(hidden_dim2 // 2)
        
    def forward(self, x, edge_index):
        # First graph convolution
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        # Second graph convolution
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        # Fully connected layers
        x = self.fc1(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        # Output layer
        x = self.fc2(x)
        x = torch.sigmoid(x)
        
        return x

print("GNN Model Architecture defined!")

## 5. Data Preparation for PyTorch Geometric

In [ ]:
# Convert to PyTorch Geometric format
print("Converting to PyTorch Geometric format...")

# Create node mapping
node_list = list(G.nodes())
node_to_idx = {node_id: idx for idx, node_id in enumerate(node_list)}

# Prepare node features
features = ['traffic_speed', 'rainfall', 'road_capacity']
X = traffic_nodes_df[features].values

# Normalize features
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)
x = torch.tensor(X_normalized, dtype=torch.float32)

# Prepare edge index
edge_indices = []
for idx, row in edges_df.iterrows():
    source = row['source']
    target = row['target']
    
    if source in node_to_idx and target in node_to_idx:
        source_idx = node_to_idx[source]
        target_idx = node_to_idx[target]
        edge_indices.append([source_idx, target_idx])
        edge_indices.append([target_idx, source_idx])  # Undirected

edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()

# Prepare target labels
y = torch.tensor(traffic_nodes_df['congestion_score'].values, dtype=torch.float32).unsqueeze(1)

# Create PyTorch Geometric Data object
data = Data(x=x, edge_index=edge_index, y=y, num_nodes=len(node_list))

print(f"PyTorch Geometric Data created:")
print(f"- Nodes: {data.num_nodes:,}")
print(f"- Edges: {data.num_edges:,}")
print(f"- Features: {data.num_node_features}")
print(f"- Target shape: {data.y.shape}")

## 6. Model Training

In [ ]:
# Training setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TrafficGNN(input_dim=3, hidden_dim1=16, hidden_dim2=8, output_dim=1).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

# Train/validation split
num_nodes = data.num_nodes
indices = np.random.permutation(num_nodes)
split_idx = int(num_nodes * 0.8)

train_indices = indices[:split_idx]
val_indices = indices[split_idx:]

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[train_indices] = True
val_mask[val_indices] = True

data.train_mask = train_mask
data.val_mask = val_mask

print(f"Training nodes: {train_mask.sum().item():,}")
print(f"Validation nodes: {val_mask.sum().item():,}")
print(f"Device: {device}")

In [ ]:
# Training loop
epochs = 50
train_losses = []
val_losses = []
best_val_loss = float('inf')
patience = 15
epochs_no_improve = 0

print("Starting training...")

for epoch in range(epochs):
    # Training
    model.train()
    optimizer.zero_grad()
    
    out = model(data.x.to(device), data.edge_index.to(device))
    train_loss = criterion(out[data.train_mask], data.y.to(device)[data.train_mask])
    train_loss.backward()
    optimizer.step()
    
    train_losses.append(train_loss.item())
    
    # Validation
    model.eval()
    with torch.no_grad():
        val_out = model(data.x.to(device), data.edge_index.to(device))
        val_loss = criterion(val_out[data.val_mask], data.y.to(device)[data.val_mask])
        val_losses.append(val_loss.item())
    
    scheduler.step(val_loss)
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
    
    # Print progress
    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f'Epoch {epoch+1:3d}/{epochs:3d} | Train Loss: {train_loss.item():.4f} | Val Loss: {val_loss.item():.4f}')
    
    # Early stopping
    if epochs_no_improve >= patience:
        print(f'Early stopping at epoch {epoch+1}')
        break

# Load best model
model.load_state_dict(best_model_state)
print(f'\nTraining completed! Best validation loss: {best_val_loss:.4f}')

In [ ]:
# Plot training history
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training History', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../output/training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("Training history plot saved!")

## 7. Model Evaluation & Predictions

In [ ]:
# Generate predictions
print("Generating predictions...")
model.eval()
with torch.no_grad():
    predictions = model(data.x.to(device), data.edge_index.to(device))
    predictions = predictions.cpu().numpy().flatten()

# Calculate metrics
actual = data.y.cpu().numpy().flatten()
mse = mean_squared_error(actual, predictions)
mae = mean_absolute_error(actual, predictions)
r2 = r2_score(actual, predictions)
rmse = np.sqrt(mse)

print("\nModel Performance Metrics:")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R²: {r2:.4f}")

print(f"\nPrediction Summary:")
print(f"Average predicted congestion: {np.mean(predictions):.4f}")
print(f"Average actual congestion: {np.mean(actual):.4f}")
print(f"Prediction range: [{np.min(predictions):.4f}, {np.max(predictions):.4f}]")

In [ ]:
# Create predictions dataframe
predictions_df = traffic_nodes_df.copy()
predictions_df['predicted_congestion'] = predictions
predictions_df['prediction_error'] = predictions - predictions_df['congestion_score']

# Get top congested nodes
top_congested = predictions_df.nlargest(10, 'predicted_congestion')

print("\nTop 10 Most Congested Nodes:")
print(top_congested[['node_id', 'predicted_congestion', 'actual_congestion', 
                    'traffic_speed', 'rainfall', 'road_capacity']].to_string(index=False))

# Save predictions
predictions_df.to_csv('../output/traffic_predictions.csv', index=False)
top_congested.to_csv('../output/top_congested_nodes.csv', index=False)
print("\nPredictions saved successfully!")

## 8. Interactive Visualization

In [ ]:
# Create interactive heatmap
print("Creating interactive traffic heatmap...")

# Base map
bangalore_coords = [12.9716, 77.5946]
m = folium.Map(location=bangalore_coords, zoom_start=12, tiles='OpenStreetMap')

# Prepare heat data
heat_data = []
for idx, row in predictions_df.iterrows():
    if not pd.isna(row['y']) and not pd.isna(row['x']):
        intensity = row['predicted_congestion']
        heat_data.append([row['y'], row['x'], intensity])

# Add heatmap
if heat_data:
    HeatMap(heat_data, 
           min_opacity=0.2,
           max_zoom=18,
           radius=15,
           blur=10,
           gradient={0.2: 'green', 0.4: 'yellow', 0.6: 'orange', 0.8: 'red', 1.0: 'darkred'}
          ).add_to(m)

# Add title
title_html = '''
<h3 align="center" style="font-size:20px"><b>Bangalore Traffic Congestion Heatmap</b></h3>
'''
m.get_root().html.add_child(folium.Element(title_html))

# Save map
m.save('../output/traffic_heatmap.html')
print("Traffic heatmap saved!")

# Display map (if running in Jupyter)
display(m)

In [ ]:
# Create prediction analysis plots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Predicted vs Actual
axes[0, 0].scatter(predictions_df['congestion_score'], 
                  predictions_df['predicted_congestion'], 
                  alpha=0.5, s=1)
axes[0, 0].plot([0, 1], [0, 1], 'r--', label='Perfect Prediction')
axes[0, 0].set_xlabel('Actual Congestion')
axes[0, 0].set_ylabel('Predicted Congestion')
axes[0, 0].set_title('Predicted vs Actual Congestion')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Error Distribution
errors = predictions_df['predicted_congestion'] - predictions_df['congestion_score']
axes[0, 1].hist(errors, bins=50, alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Prediction Error')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Prediction Error Distribution')
axes[0, 1].grid(True, alpha=0.3)

# Congestion Distribution
axes[1, 0].hist(predictions_df['predicted_congestion'], bins=50, alpha=0.7, 
               label='Predicted', edgecolor='black')
axes[1, 0].hist(predictions_df['congestion_score'], bins=50, alpha=0.7, 
               label='Actual', edgecolor='black')
axes[1, 0].set_xlabel('Congestion Score')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Congestion Score Distribution')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Features vs Congestion
axes[1, 1].scatter(predictions_df['traffic_speed'], 
                  predictions_df['predicted_congestion'], 
                  alpha=0.5, s=1, label='Speed')
axes[1, 1].set_xlabel('Traffic Speed (km/h)')
axes[1, 1].set_ylabel('Predicted Congestion')
axes[1, 1].set_title('Traffic Speed vs Predicted Congestion')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/prediction_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Analysis plots saved!")

## 9. Summary & Next Steps

In [ ]:
# Project summary
print("="*80)
print("BENGALURU BRAIN: M1 TRAFFIC MODULE - PIPELINE SUMMARY")
print("="*80)

print("\n📊 DATA STATISTICS:")
print(f"   • Road network nodes: {G.number_nodes():,}")
print(f"   • Road network edges: {G.number_edges():,}")
print(f"   • Traffic features: 3 (speed, rainfall, capacity)")
print(f"   • Target variable: congestion score (0-1)")

print("\n🤖 MODEL ARCHITECTURE:")
print(f"   • Model type: Graph Convolutional Network (GCN)")
print(f"   • Input features: 3")
print(f"   • Hidden layers: 16 → 8")
print(f"   • Output: 1 (congestion score)")
print(f"   • Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

print("\n📈 PERFORMANCE METRICS:")
print(f"   • MSE: {mse:.4f}")
print(f"   • RMSE: {rmse:.4f}")
print(f"   • MAE: {mae:.4f}")
print(f"   • R²: {r2:.4f}")

print("\n📁 OUTPUT FILES GENERATED:")
files = [
    "bangalore.graphml",
    "traffic_graph.pt", 
    "traffic_model.pth",
    "traffic_predictions.csv",
    "traffic_heatmap.html",
    "training_history.png",
    "prediction_analysis.png"
]
for file in files:
    print(f"   • {file}")

print("\n🚀 NEXT STEPS FOR PHASE 3 & 4:")
print("   1. Add IT Shift Dataset features")
print("   2. Integrate external civic data (IMD rainfall, KSNDMC flood risk)")
print("   3. Upgrade to STGCN/GraphWaveNet for temporal predictions")
print("   4. Target: >75% accuracy, <20% false positives, <2min inference")

print("\n✅ PIPELINE COMPLETED SUCCESSFULLY!")
print("="*80)